# 06_semantic_and_cost_routing

In enterprise AI engineering, routing is where massive FinOps optimization happens. Sending every user query—from simple greetings and format parsing to complex code generation—to a frontier model (gpt-4o or claude-3-5-sonnet) causes API costs to spiral out of control.

Production systems implement Model Routing to dynamically evaluate incoming task complexity and route low-complexity queries to fast, cheap models (gpt-4o-mini, llama-3-8b), reserving heavy models only when true reasoning is required.


## 1. Core Concepts & Routing Paradigms

The Pareto Principle of LLM Workloads: Industry benchmarks consistently show that roughly 80% to 85% of standard application queries (e.g., standard classification, entity extraction, formatting, straightforward Q&A) can be fully satisfied by lightweight models, while only 15% require frontier intelligence.

### Routing Architectures:

**Heuristic Routers:** Rule-based dispatch using string length, regex patterns, or keyword detection (fast, zero cost, but rigid).
**Classifier/Semantic Routers:** Embedding incoming prompts and checking cosine similarity against pre-defined reference categories, or using a tiny local classifier (like ModernBERT) to decide the destination model with minimal latency overhead.  
**Cheap-First / Cascading Routers:** Sending the request to a cheap model first, evaluating a confidence score or verification token check, and dynamically escalating to an expensive frontier model only if the cheap model fails or returns uncertainty.  

## 2. Production Implementation Code (Cost-Aware Dynamic Model Router)
Here is a clean architectural implementation of a runtime router that inspects prompt traits and dispatches execution across multiple tiers.

In [ ]:
import os
from openai import OpenAI

def evaluate_complexity_heuristic(prompt: str) -> str:
    """
    Evaluates prompt complexity using lightweight heuristics.
    In advanced setups, this can be replaced by a local embedding or a small BERT classifier.
    """
    word_count = len(prompt.split())
    complex_keywords = ["refactor", "architect", "debug", "proof", "multistep", "algorithm"]
    
    # Check for complex indicators
    has_complex_keyword = any(kw in prompt.lower() for kw in complex_keywords)
    
    if word_count < 15 and not has_complex_keyword:
        return "fast-tier" # Route to cheap model
    else:
        return "frontier-tier" # Route to high-intelligence model

def smart_route_completion(prompt: str):
    """
    Gateway function that intercepts requests, routes based on intelligence requirements,
    and handles fallback routines.
    """
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    route_decision = evaluate_complexity_heuristic(prompt)
    
    # Map tiers to specific underlying model IDs
    model_map = {
        "fast-tier": "gpt-4o-mini",       # Low cost, low latency
        "frontier-tier": "gpt-4o"         # High capability, high cost
    }
    
    selected_model = model_map[route_decision]
    print(f"[Router Engine] Prompt routed to -> [{route_decision}] using model: {selected_model}")
    
    try:
        response = client.chat.completions.create(
            model=selected_model,
            messages=[
                {"role": "system", "content": "You are a helpful enterprise assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content
        
    except Exception as e:
        # Fallback loop: If primary tier fails or rate limits, fall back safely
        print(f"[Router Warning] Primary model {selected_model} failed. Executing fallback...")
        fallback_response = client.chat.completions.create(
            model="gpt-4o-mini", # Ultimate safety net model
            messages=[{"role": "user", "content": prompt}]
        )
        return fallback_response.choices[0].message.content

if __name__ == "__main__":
    print("--- Test 1: Simple Query (Expected Fast Tier) ---")
    print(smart_route_completion("What is the capital of France?"))
    
    print("\n--- Test 2: Complex Query (Expected Frontier Tier) ---")
    print(smart_route_completion("Refactor this multi-threaded Python class to prevent race conditions and prove thread safety."))

## 3. Deep-Dive: Architecture & FinOps Trade-offs

**Latency Overhead of Routers:** If your router relies on a separate API call or an embedding vector look-up phase before hitting the target model, it adds baseline latency (e.g., 20–50ms). How you balance routing precision against added time-to-first-token. Solution: Use extremely fast local CPU-bound classifiers (like HuggingFace lightweight text classifiers) or prefix-rule logic for instant decisions.  

**The Fallback / Circuit Breaker Pattern:** Production gateways must track provider health metrics. If an upstream provider starts throwing cascading 5xx errors or rate limits (429), the router must dynamically shed traffic away from that provider entirely, failing over seamlessly to secondary local instances (e.g., Ollama/vLLM pools) without impacting the end user.